# OCI Resource Management

Notebook to manage Oracle Cloud Infrastructure (OCI) resources via OCI CLI and Python SDK.

**Tenancy OCID:** `ocid1.tenancy.oc1..aaaaaaaarkr3tvxxmzwueaz3dazimmlsoqk2nc6j77vg33jinbnaupdnokxa`

## Prerequisites

1. Install OCI CLI + SDK (cell below)
2. Run `oci session authenticate` to get a session token
3. List resources

In [1]:
# Install OCI CLI and Python SDK
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'oci', 'oci-cli'])
import oci
print(f'OCI SDK version: {oci.__version__}')
result = subprocess.run([sys.executable, '-m', 'oci_cli.cli', '--version'], capture_output=True, text=True)
print(f'OCI CLI version: {result.stdout.strip() or result.stderr.strip()}')

OCI SDK version: 2.174.0
OCI CLI version: 3.82.0


## OCI Session Authentication

Run this in a terminal (not in the notebook) — it opens a browser for interactive login:

```bash
source ~/oci-venv/bin/activate
oci session authenticate --region eu-paris-1
```

This creates `~/.oci/config` with a `DEFAULT` profile and a session token valid for 1 hour.

To refresh an expired session:
```bash
oci session refresh --profile DEFAULT
```

In [4]:
# Configuration — Session Token Auth
import oci
import json
from pathlib import Path

TENANCY_OCID = 'ocid1.tenancy.oc1..aaaaaaaarkr3tvxxmzwueaz3dazimmlsoqk2nc6j77vg33jinbnaupdnokxa'
REGION = 'eu-paris-1'

# Load OCI config from session token auth
config_path = str(Path.home() / '.oci' / 'config')
try:
    config = oci.config.from_file(config_path, 'DEFAULT')
    config['region'] = REGION
    # Session token auth requires SecurityTokenSigner
    token_file = config.get('security_token_file')
    if token_file:
        with open(token_file) as f:
            token = f.read().strip()
        private_key = oci.signer.load_private_key_from_file(config['key_file'])
        signer = oci.auth.signers.SecurityTokenSigner(token, private_key)
        print(f'Session token loaded: region={REGION}')
        print(f'Tenancy: {config["tenancy"][:50]}...')
    else:
        signer = oci.signer.Signer.from_config(config)
        print(f'API Key auth loaded: region={REGION}')
except Exception as e:
    signer = None
    print(f'ERROR: {e}')
    print('Run `oci session authenticate --region eu-paris-1` in a terminal first.')

Session token loaded: region=eu-paris-1
Tenancy: ocid1.tenancy.oc1..aaaaaaaarkr3tvxxmzwueaz3dazimml...


In [5]:
# Validate session — list regions available to this tenancy
identity_client = oci.identity.IdentityClient(config, signer=signer)
regions = identity_client.list_region_subscriptions(TENANCY_OCID).data
print('Subscribed regions:')
for r in regions:
    print(f'  {r.region_name} (home={r.is_home_region})')

Subscribed regions:
  eu-paris-1 (home=False)
  eu-frankfurt-1 (home=True)


/home/ga1/msftmh/.venv-1/lib/python3.12/site-packages/urllib3/poolmanager.py:329: FutureWarning: The 'strict' parameter is no longer needed on Python 3+. This will raise an error in urllib3 v3.0.
  warnings.warn(


## List All Compartments

In [6]:
# List all compartments in the tenancy
identity_client = oci.identity.IdentityClient(config, signer=signer)
compartments = oci.pagination.list_call_get_all_results(
    identity_client.list_compartments,
    TENANCY_OCID,
    compartment_id_in_subtree=True,
    access_level='ACCESSIBLE'
).data

print(f'Found {len(compartments)} compartments:\n')
for c in compartments:
    state = c.lifecycle_state
    print(f'  [{state:8s}] {c.name}')
    print(f'             OCID: {c.id}')
    if c.description:
        print(f'             Desc: {c.description}')
    print()

Found 9 compartments:

  [ACTIVE  ] 4aecf0e8-2fe2-4187-bc93-0356bd2676f5
             OCID: ocid1.compartment.oc1..aaaaaaaayehuog6myqxudqejx3ddy6bzkr2f3dnjuuygs424taimn4av4wbq
             Desc: Mapped external cloud mapped entity ID

  [ACTIVE  ] anchorodaa_anchorodaa_202601230727
             OCID: ocid1.compartment.oc1..aaaaaaaaucrhusiyndkss5wcor7g3sxorp6dnsfchx6qojg4ywmh5rp3mj2a
             Desc: Multicloud Resource Anchor Compartment. Managed by Oracle.

  [ACTIVE  ] anchorodaa_anchorodaa_202603051108
             OCID: ocid1.compartment.oc1..aaaaaaaabcpbmu3tgctsvwc4x5tymbbmglah4yssknjnlaf42pvebtuf3s3a
             Desc: Multicloud Resource Anchor Compartment. Managed by Oracle.

  [ACTIVE  ] anchorodaa_anchorodaa_202603051154
             OCID: ocid1.compartment.oc1..aaaaaaaaxgykufgf2vzjdygpwrqvhms34gl2mjrio2gtxspbdz2s56rityga
             Desc: Multicloud Resource Anchor Compartment. Managed by Oracle.

  [ACTIVE  ] MulticloudLink_ODBAA_20251026140711
             OCID: ocid1.c

## List All Database Resources

Search across all compartments for Oracle database resources.

In [7]:
# List Autonomous Databases across all compartments
db_client = oci.database.DatabaseClient(config, signer=signer)

all_adbs = []
search_compartments = [TENANCY_OCID] + [c.id for c in compartments if c.lifecycle_state == 'ACTIVE']

for comp_id in search_compartments:
    try:
        adbs = oci.pagination.list_call_get_all_results(
            db_client.list_autonomous_databases,
            compartment_id=comp_id
        ).data
        all_adbs.extend(adbs)
    except oci.exceptions.ServiceError as e:
        if e.status != 404:
            print(f'  Warning: {e.message[:80]}')

print(f'Found {len(all_adbs)} Autonomous Databases:\n')
for db in all_adbs:
    print(f'  [{db.lifecycle_state:12s}] {db.display_name}')
    print(f'                 OCID: {db.id}')
    print(f'                 Compartment: {db.compartment_id}')
    print(f'                 DB Version: {db.db_version}')
    print()

Found 0 Autonomous Databases:



In [10]:
# List DB Systems (BaseDB) across all compartments
all_dbsystems = []

for comp_id in search_compartments:
    try:
        dbsystems = oci.pagination.list_call_get_all_results(
            db_client.list_db_systems,
            compartment_id=comp_id
        ).data
        all_dbsystems.extend(dbsystems)
    except oci.exceptions.ServiceError as e:
        if e.status != 404:
            print(f'  Warning: {e.message[:80]}')

print(f'Found {len(all_dbsystems)} DB Systems:\n')
for ds in all_dbsystems:
    print(f'  [{ds.lifecycle_state:12s}] {ds.display_name}')
    print(f'                 OCID: {ds.id}')
    print(f'                 Compartment: {ds.compartment_id}')
    print()

Found 0 DB Systems:



In [11]:
# List Cloud VM Clusters across all compartments
all_clusters = []

for comp_id in search_compartments:
    try:
        clusters = oci.pagination.list_call_get_all_results(
            db_client.list_cloud_vm_clusters,
            compartment_id=comp_id
        ).data
        all_clusters.extend(clusters)
    except oci.exceptions.ServiceError as e:
        if e.status != 404:
            print(f'  Warning: {e.message[:80]}')

print(f'Found {len(all_clusters)} Cloud VM Clusters:\n')
for cl in all_clusters:
    print(f'  [{cl.lifecycle_state:12s}] {cl.display_name}')
    print(f'                 OCID: {cl.id}')
    print(f'                 Compartment: {cl.compartment_id}')
    print()

Found 0 Cloud VM Clusters:



In [13]:
# List Virtual Cloud Networks (VCNs) across all compartments
vcn_client = oci.core.VirtualNetworkClient(config, signer=signer)
all_vcns = []

for comp_id in search_compartments:
    try:
        vcns = oci.pagination.list_call_get_all_results(
            vcn_client.list_vcns,
            compartment_id=comp_id
        ).data
        all_vcns.extend(vcns)
    except oci.exceptions.ServiceError as e:
        if e.status != 404:
            print(f'  Warning: {e.message[:80]}')

print(f'Found {len(all_vcns)} VCNs:\n')
for v in all_vcns:
    print(f'  [{v.lifecycle_state:12s}] {v.display_name}')
    print(f'                 OCID: {v.id}')
    print(f'                 CIDR: {v.cidr_blocks}')
    print(f'                 Compartment: {v.compartment_id}')
    print()

Found 2 VCNs:

  [AVAILABLE   ] VCN-multicloudnetworklink20260421103333
                 OCID: ocid1.vcn.oc1.eu-paris-1.amaaaaaacozjngia6ridktx6tr2zdoynrhfpq2ihs7ijz2ggtpm3kjylvaha
                 CIDR: ['192.168.0.0/24']
                 Compartment: ocid1.compartment.oc1..aaaaaaaayehuog6myqxudqejx3ddy6bzkr2f3dnjuuygs424taimn4av4wbq

  [AVAILABLE   ] VCN-multicloudnetworklink20260307093017
                 OCID: ocid1.vcn.oc1.eu-paris-1.amaaaaaacozjngia5gzu5dqo5qlm6vxuw2qpvjtysfvx7r4yqsj4b3el3oza
                 CIDR: ['192.168.0.0/24']
                 Compartment: ocid1.compartment.oc1..aaaaaaaayehuog6myqxudqejx3ddy6bzkr2f3dnjuuygs424taimn4av4wbq



In [14]:
# List Network Security Groups (NSGs) across all compartments
all_nsgs = []

for comp_id in search_compartments:
    try:
        nsgs = oci.pagination.list_call_get_all_results(
            vcn_client.list_network_security_groups,
            compartment_id=comp_id
        ).data
        all_nsgs.extend(nsgs)
    except oci.exceptions.ServiceError as e:
        if e.status != 404:
            print(f'  Warning: {e.message[:80]}')

print(f'Found {len(all_nsgs)} Network Security Groups:\n')
for nsg in all_nsgs:
    print(f'  [{nsg.lifecycle_state:12s}] {nsg.display_name}')
    print(f'                 OCID: {nsg.id}')
    print(f'                 VCN:  {nsg.vcn_id}')
    print(f'                 Compartment: {nsg.compartment_id}')
    print()

Found 4 Network Security Groups:

  [AVAILABLE   ] otto2_NSG
                 OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaahdcsrhvod7ez4oytaamipzaxpk3bycy5icm7s5hsesypfhxqsb6q
                 VCN:  ocid1.vcn.oc1.eu-paris-1.amaaaaaacozjngia6ridktx6tr2zdoynrhfpq2ihs7ijz2ggtpm3kjylvaha
                 Compartment: ocid1.compartment.oc1..aaaaaaaayehuog6myqxudqejx3ddy6bzkr2f3dnjuuygs424taimn4av4wbq

  [AVAILABLE   ] myoracle2azuredb_NSG
                 OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaazlfnigd2e3hgdkpsqszulpwd3slsrndhxhglzgzrd6vm2vrfgizq
                 VCN:  ocid1.vcn.oc1.eu-paris-1.amaaaaaacozjngia6ridktx6tr2zdoynrhfpq2ihs7ijz2ggtpm3kjylvaha
                 Compartment: ocid1.compartment.oc1..aaaaaaaayehuog6myqxudqejx3ddy6bzkr2f3dnjuuygs424taimn4av4wbq

  [AVAILABLE   ] ClarkKent01_NSG
                 OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaaxtof2b4ano6tbqzduofdszgkzutcv7urmfjvqdcfbuf62ouyw65q
                 VCN:  ocid1.vcn.oc1.eu-pa

## Summary

In [15]:
# Summary of all resources found
print('=' * 60)
print('OCI Resource Summary')
print('=' * 60)
print(f'Tenancy:              {TENANCY_OCID[:40]}...')
print(f'Region:               {REGION}')
print(f'Compartments:         {len(compartments)}')
print(f'Autonomous DBs:       {len(all_adbs)}')
print(f'DB Systems (BaseDB):  {len(all_dbsystems)}')
print(f'Cloud VM Clusters:    {len(all_clusters)}')
print(f'VCNs:                 {len(all_vcns)}')
print(f'Network Sec. Groups:  {len(all_nsgs)}')
print('=' * 60)

total = len(all_adbs) + len(all_dbsystems) + len(all_clusters) + len(all_vcns) + len(all_nsgs)
if total == 0:
    print('\n✅ No resources found — tenancy is clean.')
else:
    print(f'\n⚠️  {total} resource(s) found — review before cleanup.')

OCI Resource Summary
Tenancy:              ocid1.tenancy.oc1..aaaaaaaarkr3tvxxmzwue...
Region:               eu-paris-1
Compartments:         9
Autonomous DBs:       0
DB Systems (BaseDB):  0
Cloud VM Clusters:    0
VCNs:                 2
Network Sec. Groups:  4

⚠️  6 resource(s) found — review before cleanup.


## Cleanup

Delete orphaned networking resources. Order: NSGs first, then VCNs (NSGs depend on VCNs).

> **Set `DRY_RUN = False`** to actually delete resources.

In [17]:
DRY_RUN = False  # Set to True for dry run

import time

vcn_client = oci.core.VirtualNetworkClient(config, signer=signer)

# --- Step 1: Delete NSGs ---
print('=== Deleting Network Security Groups ===\n')
for nsg in all_nsgs:
    print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting NSG: {nsg.display_name}')
    print(f'    OCID: {nsg.id}')
    if not DRY_RUN:
        try:
            vcn_client.delete_network_security_group(nsg.id)
            print(f'    ✅ Deleted')
        except oci.exceptions.ServiceError as e:
            print(f'    ❌ Error: {e.message[:100]}')
    print()

# --- Step 2: Delete Subnets (required before VCN deletion) ---
print('=== Deleting Subnets ===\n')
for v in all_vcns:
    try:
        subnets = oci.pagination.list_call_get_all_results(
            vcn_client.list_subnets,
            compartment_id=v.compartment_id,
            vcn_id=v.id
        ).data
        for sn in subnets:
            print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting Subnet: {sn.display_name}')
            print(f'    OCID: {sn.id}')
            if not DRY_RUN:
                try:
                    vcn_client.delete_subnet(sn.id)
                    print(f'    ✅ Deleted')
                except oci.exceptions.ServiceError as e:
                    print(f'    ❌ Error: {e.message[:100]}')
            print()
    except oci.exceptions.ServiceError as e:
        print(f'  Warning listing subnets for VCN {v.display_name}: {e.message[:100]}')

# --- Step 3: Delete Internet/NAT/Service Gateways and Route Tables ---
print('=== Deleting Gateways & Route Tables ===\n')
for v in all_vcns:
    # Internet Gateways
    try:
        igws = vcn_client.list_internet_gateways(compartment_id=v.compartment_id, vcn_id=v.id).data
        for igw in igws:
            print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting Internet GW: {igw.display_name} (OCID: {igw.id})')
            if not DRY_RUN:
                try:
                    vcn_client.delete_internet_gateway(igw.id)
                    print(f'    ✅ Deleted')
                except oci.exceptions.ServiceError as e:
                    print(f'    ❌ Error: {e.message[:100]}')
    except oci.exceptions.ServiceError:
        pass

    # NAT Gateways
    try:
        nats = vcn_client.list_nat_gateways(compartment_id=v.compartment_id, vcn_id=v.id).data
        for nat in nats:
            print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting NAT GW: {nat.display_name} (OCID: {nat.id})')
            if not DRY_RUN:
                try:
                    vcn_client.delete_nat_gateway(nat.id)
                    print(f'    ✅ Deleted')
                except oci.exceptions.ServiceError as e:
                    print(f'    ❌ Error: {e.message[:100]}')
    except oci.exceptions.ServiceError:
        pass

    # Service Gateways
    try:
        sgws = vcn_client.list_service_gateways(compartment_id=v.compartment_id, vcn_id=v.id).data
        for sgw in sgws:
            print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting Service GW: {sgw.display_name} (OCID: {sgw.id})')
            if not DRY_RUN:
                try:
                    vcn_client.delete_service_gateway(sgw.id)
                    print(f'    ✅ Deleted')
                except oci.exceptions.ServiceError as e:
                    print(f'    ❌ Error: {e.message[:100]}')
    except oci.exceptions.ServiceError:
        pass

    # Non-default Route Tables
    try:
        rts = vcn_client.list_route_tables(compartment_id=v.compartment_id, vcn_id=v.id).data
        for rt in rts:
            if rt.id != v.default_route_table_id:
                print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting Route Table: {rt.display_name} (OCID: {rt.id})')
                if not DRY_RUN:
                    try:
                        vcn_client.delete_route_table(rt.id)
                        print(f'    ✅ Deleted')
                    except oci.exceptions.ServiceError as e:
                        print(f'    ❌ Error: {e.message[:100]}')
    except oci.exceptions.ServiceError:
        pass

    # Non-default Security Lists
    try:
        sls = vcn_client.list_security_lists(compartment_id=v.compartment_id, vcn_id=v.id).data
        for sl in sls:
            if sl.id != v.default_security_list_id:
                print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting Security List: {sl.display_name} (OCID: {sl.id})')
                if not DRY_RUN:
                    try:
                        vcn_client.delete_security_list(sl.id)
                        print(f'    ✅ Deleted')
                    except oci.exceptions.ServiceError as e:
                        print(f'    ❌ Error: {e.message[:100]}')
    except oci.exceptions.ServiceError:
        pass
    print()

# --- Step 4: Delete VCNs ---
print('=== Deleting VCNs ===\n')
for v in all_vcns:
    print(f'  {"[DRY RUN] " if DRY_RUN else ""}Deleting VCN: {v.display_name}')
    print(f'    OCID: {v.id}')
    if not DRY_RUN:
        try:
            vcn_client.delete_vcn(v.id)
            print(f'    ✅ Deleted')
        except oci.exceptions.ServiceError as e:
            print(f'    ❌ Error: {e.message[:100]}')
    print()

if DRY_RUN:
    print('🔒 DRY RUN complete — no resources were deleted.')
    print('   Set DRY_RUN = False and re-run to delete.')
else:
    print('✅ Cleanup complete.')

=== Deleting Network Security Groups ===

  Deleting NSG: otto2_NSG
    OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaahdcsrhvod7ez4oytaamipzaxpk3bycy5icm7s5hsesypfhxqsb6q
    ❌ Error: The required information to complete authentication was not provided or was incorrect.

  Deleting NSG: myoracle2azuredb_NSG
    OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaazlfnigd2e3hgdkpsqszulpwd3slsrndhxhglzgzrd6vm2vrfgizq
    ❌ Error: The required information to complete authentication was not provided or was incorrect.

  Deleting NSG: ClarkKent01_NSG
    OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaaxtof2b4ano6tbqzduofdszgkzutcv7urmfjvqdcfbuf62ouyw65q
    ❌ Error: The required information to complete authentication was not provided or was incorrect.

  Deleting NSG: adbuser00_NSG
    OCID: ocid1.networksecuritygroup.oc1.eu-paris-1.aaaaaaaastiil6pbmlqyvqmb25y5vmmlagrrdvs2io24rxmor66mc7vj4ryq
    ❌ Error: The required information to complete authentication was not provi